# Climate data handling

## Save future climate data from CBP

In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import os

s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
bucket = 'bluefish-datashare.example' # set to real path

stations = ['N36017', 'N51001', 'N24013', 'N51820']
prefixes = [
    'P6_climate/FutureHydroForcing/HPRC/',
    'P6_climate/FutureHydroForcing/HTMP/'
]

out_dir = # where to save climate data
os.makedirs(out_dir, exist_ok=True)

paginator = s3.get_paginator('list_objects_v2')
for prefix in prefixes:
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get('Contents', []):
            key = obj['Key']
            if any(f'{st}.' in key for st in stations):
                filename = key.replace('/', '_')
                local_path = os.path.join(out_dir, filename)
                print(f'Downloading: {key}')
                s3.download_file(bucket, key, local_path)

In [10]:
available_stations = set()
for page in paginator.paginate(Bucket=bucket, Prefix='P6_climate/FutureHydroForcing/HPRC/'):
    for obj in page.get('Contents', []):
        key = obj['Key']
        filename = key.split('/')[-1]
        if filename.endswith('.PRC'):
            station = filename.replace('.PRC', '')
            available_stations.add(station)

print(sorted(available_stations))

['N10001', 'N24013', 'N24039', 'N24043', 'N36017', 'N42021', 'N42087', 'N51001', 'N51145', 'N51820']


## Pre-process CBP data

In [ ]:
# break up filename to get RCP, region, period, and datatype
#syntax is phase_climate_futureydroforcing_datatype_modelrcp_kk_P50_startyear-endyear_regionid.datatype2
# ex: P6_climate_FutureHydroForcing_HPRC_HBD45_KK_P50_2031-2040_N24013
# datatype is either HRPC (=precip) or HTMP (=temperature)
# model is HBD45, PRSIM, RCP45, or RCP85
# start-endyear
# regionid is N24013 (piedmont), N36017 (appalachia), N51001 (coastal plain), or N51820 (valley and ridge)

import os
import pandas as pd

raw_data_path = # enter path to extracted climate data 
output_path = # where to save processed csvs

# map region to land segment id
region_map = {
    "N24013": ("piedmont", "p"),
    "N36017": ("appalachia", "a"),
    "N51001": ("coastal_plain", "cp"),
    "N51820": ("valley_ridge", "vr")
}

# pair period, precip_model, temp_model for output_name
pairings = [
    ("2021-2030", "PRISM_KK_P50", "RCP45_P50", "PRISM_RCP45_2021-2030"),
    ("2031-2040", "HBD45_KK_P50", "RCP45_P50", "HBD45_RCP45_2031-2040"),
    ("2041-2050", "HBD45_KK_P50", "RCP45_P50", "HBD45_RCP45_2041-2050"),
    ("2051-2060", "RCP45_KK_P50", "RCP45_P50", "RCP45_2051-2060"),
    ("2061-2070", "RCP45_KK_P50", "RCP45_P50", "RCP45_2061-2070"),
    ("2071-2080", "RCP45_KK_P50", "RCP45_P50", "RCP45_2071-2080"),
    ("2081-2090", "RCP45_KK_P50", "RCP45_P50", "RCP45_2081-2090"),
    ("2051-2060", "RCP85_KK_P50", "RCP85_P50", "RCP85_2051-2060"),
    ("2061-2070", "RCP85_KK_P50", "RCP85_P50", "RCP85_2061-2070"),
    ("2071-2080", "RCP85_KK_P50", "RCP85_P50", "RCP85_2071-2080"),
    ("2081-2090", "RCP85_KK_P50", "RCP85_P50", "RCP85_2081-2090"),
]

# process by region
for region_id, (region_name, region_abbrev) in region_map.items():
    
    # Create region output folder
    region_folder = os.path.join(output_path, region_name)
    os.makedirs(region_folder, exist_ok=True)
    
    for period, precip_model, temp_model, scenario_name in pairings:
        
        # Build filenames
        precip_filename = f"P6_climate_FutureHydroForcing_HPRC_{precip_model}_{period}_{region_id}.PRC"
        temp_filename = f"P6_climate_FutureHydroForcing_HTMP_{temp_model}_{period}_{region_id}.TMP"
        
        precip_file = os.path.join(raw_data_path, precip_filename)
        temp_file = os.path.join(raw_data_path, temp_filename)
        
        # Check files exist
        if not os.path.exists(precip_file):
            print(f"Missing precip: {precip_filename}")
            continue
        if not os.path.exists(temp_file):
            print(f"Missing temp: {temp_filename}")
            continue
        
        # Read files
        precip = pd.read_csv(precip_file, header=None, names=["year", "month", "day", "hour", "precip"])
        temp = pd.read_csv(temp_file, header=None, names=["year", "month", "day", "hour", "temp"])
        
        # Merge on date/time columns
        combined = pd.merge(precip, temp, on=["year", "month", "day", "hour"])
        
        # Convert precip from inches to mm
        combined["precip"] = combined["precip"] * 25.4
        
        # Update years (1991 -> start year)
        start_year = int(period.split("-")[0])
        combined["year"] = combined["year"] - 1991 + start_year
        
        # Output file
        output_file = os.path.join(region_folder, f"{region_abbrev}_{scenario_name}.csv")
        combined.to_csv(output_file, index=False)
        
        print(f"Created: {output_file}")


## Update weather files

In [ ]:
import os
import shutil
import pandas as pd
import numpy as np
from tqdm import tqdm

# fill paths
baseline_folder = # path to historic runs folder
combined_csv_folder = # path to processed csvs (from step above)
output_parent = # path to outputted weather files

# mapping
region_abbrev_to_climate_prefix = {
    "a": "appalachia",
    "cp": "coastal",
    "p": "piedmont",
    "vr": "valley"
}

region_abbrev_to_csv_folder = {
    "a": "appalachia",
    "cp": "coastal_plain",
    "p": "piedmont",
    "vr": "valley_ridge"
}

# Scenarios to process (must match CSV filenames from Stage 1)
scenarios = [
    "PRISM_RCP45_2021-2030",
    "HBD45_RCP45_2031-2040",
    "HBD45_RCP45_2041-2050",
    "RCP45_2051-2060",
    "RCP45_2061-2070",
    "RCP45_2071-2080",
    "RCP45_2081-2090",
    "RCP85_2051-2060",
    "RCP85_2061-2070",
    "RCP85_2071-2080",
    "RCP85_2081-2090",
]

# Helper functions

def get_region_abbrev(folder_name):
    """Extract region abbreviation from folder name (e.g., 'a-rc-corn' -> 'a')"""
    for abbrev in ["cp", "vr", "a", "p"]:  # check cp/vr first (longer matches)
        if folder_name.startswith(abbrev + "-"):
            return abbrev
    return None

def load_combined_csv(csv_folder, region_abbrev, scenario_name):
    """Load the combined climate CSV for a region and scenario"""
    filename = f"{region_abbrev}_{scenario_name}.csv"
    region_folder = region_abbrev_to_csv_folder[region_abbrev]
    filepath = os.path.join(csv_folder, region_folder, filename)
    return pd.read_csv(filepath)

def update_hly_file(hly_path, future_df):
    """
    Update .hly file: replace 1990-1999 precip with future data, keep 1980-1989
    """
    # Read existing file
    with open(hly_path, 'r') as f:
        lines = f.readlines()
    
    # Parse future data - group by year, month, day, hour
    future_precip = {}
    for _, row in future_df.iterrows():
        # Map future year to 1990-1999
        future_year = int(row['year'])
        mapped_year = 1990 + (future_year % 10)
        key = (mapped_year, int(row['month']), int(row['day']), int(row['hour']))
        future_precip[key] = row['precip']
    
    # Update lines for 1990-1999
    new_lines = []
    for line in lines:
        parts = line.split()
        if len(parts) >= 5:
            year = int(parts[0])
            if year >= 1990:
                month = int(parts[1])
                day = int(parts[2])
                hour = int(parts[3])
                key = (year, month, day, hour)
                if key in future_precip:
                    # Format: year(4) month(4) day(4) hour(10) precip(10)
                    new_line = f"{year:4d}{month:4d}{day:4d}{hour:10d}{future_precip[key]:10.3f}\n"
                    new_lines.append(new_line)
                else:
                    new_lines.append(line)
            else:
                new_lines.append(line)
        else:
            new_lines.append(line)
    
    # Write updated file
    with open(hly_path, 'w') as f:
        f.writelines(new_lines)

def update_dly_file(dly_path, future_df):
    """
    Update .dly file: replace 1990-1999 tmax/tmin with daily min/max from hourly temp
    Keep all other columns unchanged
    Format: year(6) month(4) day(4) rad(6) tmax(6) tmin(6) precip(6) rhum(6) wspd(6)
    """
    # Calculate daily tmax/tmin from hourly data
    future_df = future_df.copy()
    future_df['mapped_year'] = 1990 + (future_df['year'] % 10)
    daily_temp = future_df.groupby(['mapped_year', 'month', 'day']).agg(
        tmax=('temp', 'max'),
        tmin=('temp', 'min')
    ).reset_index()
    
    # Create lookup dict
    temp_lookup = {}
    for _, row in daily_temp.iterrows():
        key = (int(row['mapped_year']), int(row['month']), int(row['day']))
        temp_lookup[key] = (row['tmax'], row['tmin'])
    
    # Read existing file
    with open(dly_path, 'r') as f:
        lines = f.readlines()
    
    # Update lines for 1990-1999
    new_lines = []
    for line in lines:
        # Parse by fixed-width positions
        # Format: 6 chars year, 4 chars month, 4 chars day, then 6 chars each for rad/tmax/tmin/precip/rhum/wspd
        if len(line.strip()) > 0:
            try:
                year = int(line[0:6])
                if year >= 1990:
                    month = int(line[6:10])
                    day = int(line[10:14])
                    key = (year, month, day)
                    if key in temp_lookup:
                        tmax, tmin = temp_lookup[key]
                        # Keep original values for other columns
                        rad = line[14:20]      # keep as string to preserve formatting
                        precip = line[32:38]   # keep as string
                        rhum = line[38:44]     # keep as string  
                        wspd = line[44:50]     # keep as string
                        # Rebuild line with new tmax/tmin
                        new_line = f"{year:6d}{month:4d}{day:4d}{rad}{tmax:6.1f}{tmin:6.1f}{precip}{rhum}{wspd}\n"
                        new_lines.append(new_line)
                    else:
                        new_lines.append(line)
                else:
                    new_lines.append(line)
            except (ValueError, IndexError):
                new_lines.append(line)
        else:
            new_lines.append(line)
    
    # Write updated file
    with open(dly_path, 'w') as f:
        f.writelines(new_lines)

def update_wp1_file(wp1_path, future_df):
    """
    Update .WP1 file: recompute TMX, TMN, SDMX, SDMN, PRCP from future data
    """
    # Calculate daily values first
    future_df['mapped_year'] = 1990 + (future_df['year'] % 10)
    daily = future_df.groupby(['mapped_year', 'month', 'day']).agg(
        tmax=('temp', 'max'),
        tmin=('temp', 'min'),
        precip=('precip', 'sum')
    ).reset_index()
    
    # Calculate monthly statistics
    monthly = daily.groupby('month').agg(
        tmx=('tmax', 'mean'),
        tmn=('tmin', 'mean'),
        sdmx=('tmax', 'std'),
        sdmn=('tmin', 'std'),
        prcp=('precip', 'sum')
    ).reset_index()
    
    # Average across years for precip (sum was per year, need avg monthly)
    n_years = daily['mapped_year'].nunique()
    monthly['prcp'] = monthly['prcp'] / n_years
    
    # Fill NaN std with 0
    monthly = monthly.fillna(0)
    
    # Read existing file
    with open(wp1_path, 'r') as f:
        lines = f.readlines()
    
    # Update lines 3-7 (TMX, TMN, SDMX, SDMN, PRCP) - 0-indexed: 2-6
    def format_monthly_line(values, label):
        """Format 12 monthly values with label"""
        formatted = "".join(f"{v:10.2f}" for v in values)
        return formatted + f" {label}\n"
    
    # Get monthly values in order (1-12)
    monthly = monthly.sort_values('month')
    tmx_vals = monthly['tmx'].tolist()
    tmn_vals = monthly['tmn'].tolist()
    sdmx_vals = monthly['sdmx'].tolist()
    sdmn_vals = monthly['sdmn'].tolist()
    prcp_vals = monthly['prcp'].tolist()
    
    lines[2] = format_monthly_line(tmx_vals, "TMX")
    lines[3] = format_monthly_line(tmn_vals, "TMN")
    lines[4] = format_monthly_line(sdmx_vals, "SDMX")
    lines[5] = format_monthly_line(sdmn_vals, "SDMN")
    lines[6] = format_monthly_line(prcp_vals, "PRCP")
    
    # Write updated file
    with open(wp1_path, 'w') as f:
        f.writelines(lines)

def find_climate_files(scenario_path, region_abbrev):
    """Find .hly, .dly, .WP1 files in a scenario folder"""
    prefix = region_abbrev_to_climate_prefix[region_abbrev]
    
    hly_file = None
    dly_file = None
    wp1_file = None
    
    for f in os.listdir(scenario_path):
        f_lower = f.lower()
        if f_lower.startswith(prefix) and f_lower.endswith('_2su.hly'):
            hly_file = os.path.join(scenario_path, f)
        elif f_lower.startswith(prefix) and f_lower.endswith('_2su.dly'):
            dly_file = os.path.join(scenario_path, f)
        elif f_lower.startswith(prefix) and f_lower.endswith('.wp1'):
            wp1_file = os.path.join(scenario_path, f)
    
    return hly_file, dly_file, wp1_file

# MAIN PROCESSING

for scenario in tqdm(scenarios, desc="Scenarios"):
    
    # Create new folder for this scenario
    new_folder = os.path.join(output_parent, scenario)
    
    if os.path.exists(new_folder):
        tqdm.write(f"Folder exists, skipping copy: {scenario}")
    else:
        tqdm.write(f"Copying baseline to: {scenario}")
        shutil.copytree(baseline_folder, new_folder)
    

    bmp_folders = []
    for land_use in os.listdir(new_folder):
        land_use_path = os.path.join(new_folder, land_use)
        if not os.path.isdir(land_use_path):
            continue
        for region_folder in os.listdir(land_use_path):
            region_path = os.path.join(land_use_path, region_folder)
            if not os.path.isdir(region_path):
                continue
            for bmp_folder in os.listdir(region_path):
                bmp_path = os.path.join(region_path, bmp_folder)
                if os.path.isdir(bmp_path):
                    # Only process a-soy-gb and vr-ha-nt folders
                    if "a-soy-gb-" in bmp_folder or "vr-ha-nt-" in bmp_folder:
                        bmp_folders.append((land_use, region_folder, bmp_folder, bmp_path))
                        
                        

    # Cache loaded CSVs by region
    csv_cache = {}
    
    # Process all BMP folders with progress bar
    for land_use, region_folder, bmp_folder, bmp_path in tqdm(bmp_folders, desc=f"  {scenario}", leave=False):
        
        region_abbrev = get_region_abbrev(region_folder)
        if region_abbrev is None:
            tqdm.write(f"  Could not determine region for: {region_folder}")
            continue
        
        # Load combined CSV (with caching)
        if region_abbrev not in csv_cache:
            try:
                csv_cache[region_abbrev] = load_combined_csv(combined_csv_folder, region_abbrev, scenario)
            except FileNotFoundError as e:
                tqdm.write(f"  CSV not found for {region_abbrev}/{scenario}")
                continue
        
        future_df = csv_cache[region_abbrev]
        
        # Find climate files
        hly_file, dly_file, wp1_file = find_climate_files(bmp_path, region_abbrev)
        
        if hly_file and dly_file and wp1_file:
            try:
                update_hly_file(hly_file, future_df)
                update_dly_file(dly_file, future_df)
                update_wp1_file(wp1_file, future_df)
            except Exception as e:
                tqdm.write(f"  ERROR in {bmp_folder}: {e}")
        else:
            missing = []
            if not hly_file: missing.append('.hly')
            if not dly_file: missing.append('.dly')
            if not wp1_file: missing.append('.wp1')
            tqdm.write(f"  Missing files in {bmp_folder}: {missing}")


## Update weather files (T and P separately)

In [ ]:
import os
import shutil
import pandas as pd
import numpy as np
from tqdm import tqdm

# Paths
baseline_folder = # path to historic runs folder
combined_csv_folder = # path to processed csvs (from step above)
output_parent = # path to outputted weather files

# Mapping
region_abbrev_to_climate_prefix = {
    "a": "appalachia",
    "cp": "coastal",
    "p": "piedmont",
    "vr": "valley"
}

region_abbrev_to_csv_folder = {
    "a": "appalachia",
    "cp": "coastal_plain",
    "p": "piedmont",
    "vr": "valley_ridge"
}

# Scenarios to process
scenarios = [
    "PRISM_RCP45_2021-2030",
    "HBD45_RCP45_2031-2040",
    "HBD45_RCP45_2041-2050",
    "RCP45_2051-2060",
    "RCP45_2061-2070",
    "RCP45_2071-2080",
    "RCP45_2081-2090",
    "RCP85_2051-2060",
    "RCP85_2061-2070",
    "RCP85_2071-2080",
    "RCP85_2081-2090",
]

# Variants to create
variants = ["Tonly", "Ponly"]  # comment out one to test 

# Helper functions

def get_region_abbrev(folder_name):
    """Extract region abbreviation from folder name (e.g., 'a-rc-corn' -> 'a')"""
    for abbrev in ["cp", "vr", "a", "p"]:
        if folder_name.startswith(abbrev + "-"):
            return abbrev
    return None

def load_combined_csv(csv_folder, region_abbrev, scenario_name):
    """Load the combined climate CSV for a region and scenario"""
    filename = f"{region_abbrev}_{scenario_name}.csv"
    region_folder = region_abbrev_to_csv_folder[region_abbrev]
    filepath = os.path.join(csv_folder, region_folder, filename)
    return pd.read_csv(filepath)

def update_hly_file(hly_path, future_df):
    """Update .hly file: replace 1990-1999 precip with future data"""
    with open(hly_path, 'r') as f:
        lines = f.readlines()
    
    future_precip = {}
    for _, row in future_df.iterrows():
        future_year = int(row['year'])
        mapped_year = 1990 + (future_year % 10)
        key = (mapped_year, int(row['month']), int(row['day']), int(row['hour']))
        future_precip[key] = row['precip']
    
    new_lines = []
    for line in lines:
        parts = line.split()
        if len(parts) >= 5:
            year = int(parts[0])
            if year >= 1990:
                month = int(parts[1])
                day = int(parts[2])
                hour = int(parts[3])
                key = (year, month, day, hour)
                if key in future_precip:
                    new_line = f"{year:4d}{month:4d}{day:4d}{hour:10d}{future_precip[key]:10.3f}\n"
                    new_lines.append(new_line)
                else:
                    new_lines.append(line)
            else:
                new_lines.append(line)
        else:
            new_lines.append(line)
    
    with open(hly_path, 'w') as f:
        f.writelines(new_lines)

def update_dly_file(dly_path, future_df):
    """Update .dly file: replace 1990-1999 tmax/tmin only"""
    future_df = future_df.copy()
    future_df['mapped_year'] = 1990 + (future_df['year'] % 10)
    daily_temp = future_df.groupby(['mapped_year', 'month', 'day']).agg(
        tmax=('temp', 'max'),
        tmin=('temp', 'min')
    ).reset_index()
    
    temp_lookup = {}
    for _, row in daily_temp.iterrows():
        key = (int(row['mapped_year']), int(row['month']), int(row['day']))
        temp_lookup[key] = (row['tmax'], row['tmin'])
    
    with open(dly_path, 'r') as f:
        lines = f.readlines()
    
    new_lines = []
    for line in lines:
        if len(line.strip()) > 0:
            try:
                year = int(line[0:6])
                if year >= 1990:
                    month = int(line[6:10])
                    day = int(line[10:14])
                    key = (year, month, day)
                    if key in temp_lookup:
                        tmax, tmin = temp_lookup[key]
                        rad = line[14:20]
                        precip = line[26:32]
                        rhum = line[32:38]
                        wspd = line[38:44]
                        new_line = f"{year:6d}{month:4d}{day:4d}{rad}{tmax:6.1f}{tmin:6.1f}{precip}{rhum}{wspd}\n"
                        new_lines.append(new_line)
                    else:
                        new_lines.append(line)
                else:
                    new_lines.append(line)
            except (ValueError, IndexError):
                new_lines.append(line)
        else:
            new_lines.append(line)
    
    with open(dly_path, 'w') as f:
        f.writelines(new_lines)

def update_dly_file_precip_only(dly_path, future_df):
    """Update .dly file: replace 1990-1999 precip only, keep temp as-is"""
    future_df = future_df.copy()
    future_df['mapped_year'] = 1990 + (future_df['year'] % 10)
    daily_precip = future_df.groupby(['mapped_year', 'month', 'day']).agg(
        precip=('precip', 'sum')
    ).reset_index()
    
    precip_lookup = {}
    for _, row in daily_precip.iterrows():
        key = (int(row['mapped_year']), int(row['month']), int(row['day']))
        precip_lookup[key] = row['precip']
    
    with open(dly_path, 'r') as f:
        lines = f.readlines()
    
    new_lines = []
    for line in lines:
        if len(line.strip()) > 0:
            try:
                year = int(line[0:6])
                if year >= 1990:
                    month = int(line[6:10])
                    day = int(line[10:14])
                    key = (year, month, day)
                    if key in precip_lookup:
                        precip = precip_lookup[key]
                        rad = line[14:20]
                        tmax = line[20:26]
                        tmin = line[26:32]
                        rhum = line[38:44]
                        wspd = line[44:50]
                        new_line = f"{year:6d}{month:4d}{day:4d}{rad}{tmax}{tmin}{precip:6.2f}{rhum}{wspd}\n"
                        new_lines.append(new_line)
                    else:
                        new_lines.append(line)
                else:
                    new_lines.append(line)
            except (ValueError, IndexError):
                new_lines.append(line)
        else:
            new_lines.append(line)
    
    with open(dly_path, 'w') as f:
        f.writelines(new_lines)

def update_wp1_file_tonly(wp1_path, future_df):
    """Update .WP1 file: recompute TMX, TMN, SDMX, SDMN only (keep baseline PRCP)"""
    future_df = future_df.copy()
    future_df['mapped_year'] = 1990 + (future_df['year'] % 10)
    daily = future_df.groupby(['mapped_year', 'month', 'day']).agg(
        tmax=('temp', 'max'),
        tmin=('temp', 'min')
    ).reset_index()
    
    monthly = daily.groupby('month').agg(
        tmx=('tmax', 'mean'),
        tmn=('tmin', 'mean'),
        sdmx=('tmax', 'std'),
        sdmn=('tmin', 'std')
    ).reset_index()
    
    monthly = monthly.fillna(0)
    
    with open(wp1_path, 'r') as f:
        lines = f.readlines()
    
    def format_monthly_line(values, label):
        formatted = "".join(f"{v:10.2f}" for v in values)
        return formatted + f" {label}\n"
    
    monthly = monthly.sort_values('month')
    tmx_vals = monthly['tmx'].tolist()
    tmn_vals = monthly['tmn'].tolist()
    sdmx_vals = monthly['sdmx'].tolist()
    sdmn_vals = monthly['sdmn'].tolist()
    
    # Only update temp lines, keep PRCP as-is
    lines[2] = format_monthly_line(tmx_vals, "TMX")
    lines[3] = format_monthly_line(tmn_vals, "TMN")
    lines[4] = format_monthly_line(sdmx_vals, "SDMX")
    lines[5] = format_monthly_line(sdmn_vals, "SDMN")
    # lines[6] stays unchanged (baseline PRCP)
    
    with open(wp1_path, 'w') as f:
        f.writelines(lines)

def update_wp1_file_ponly(wp1_path, future_df):
    """Update .WP1 file: recompute PRCP only (keep baseline TMX, TMN, SDMX, SDMN)"""
    future_df = future_df.copy()
    future_df['mapped_year'] = 1990 + (future_df['year'] % 10)
    daily = future_df.groupby(['mapped_year', 'month', 'day']).agg(
        precip=('precip', 'sum')
    ).reset_index()
    
    monthly = daily.groupby('month').agg(
        prcp=('precip', 'sum')
    ).reset_index()
    
    n_years = daily['mapped_year'].nunique()
    monthly['prcp'] = monthly['prcp'] / n_years
    
    with open(wp1_path, 'r') as f:
        lines = f.readlines()
    
    def format_monthly_line(values, label):
        formatted = "".join(f"{v:10.2f}" for v in values)
        return formatted + f" {label}\n"
    
    monthly = monthly.sort_values('month')
    prcp_vals = monthly['prcp'].tolist()
    
    # Only update PRCP line, keep temp lines as-is
    # lines[2-5] stay unchanged (baseline TMX, TMN, SDMX, SDMN)
    lines[6] = format_monthly_line(prcp_vals, "PRCP")
    
    with open(wp1_path, 'w') as f:
        f.writelines(lines)

def find_climate_files(scenario_path, region_abbrev):
    """Find .hly, .dly, .WP1 files in a scenario folder - specifically _2su versions"""
    prefix = region_abbrev_to_climate_prefix[region_abbrev]
    
    hly_file = None
    dly_file = None
    wp1_file = None
    
    for f in os.listdir(scenario_path):
        f_lower = f.lower()
        if f_lower.startswith(prefix):
            # For .hly and .dly, specifically look for _2su version
            if '_2su.hly' in f_lower:
                hly_file = os.path.join(scenario_path, f)
            elif '_2su.dly' in f_lower:
                dly_file = os.path.join(scenario_path, f)
            elif f_lower.endswith('.wp1'):
                wp1_file = os.path.join(scenario_path, f)
    
    return hly_file, dly_file, wp1_file

# MAIN PROCESSING

for scenario in tqdm(scenarios, desc="Scenarios"):
    for variant in variants:
        scenario_name = f"{scenario}_{variant}"
        new_folder = os.path.join(output_parent, scenario_name)
        
        if os.path.exists(new_folder):
            tqdm.write(f"Folder exists, skipping copy: {scenario_name}")
        else:
            tqdm.write(f"Copying baseline to: {scenario_name}")
            shutil.copytree(baseline_folder, new_folder)
        
        # Collect BMP folders
        bmp_folders = []
        for land_use in os.listdir(new_folder):
            land_use_path = os.path.join(new_folder, land_use)
            if not os.path.isdir(land_use_path):
                continue
            for region_folder in os.listdir(land_use_path):
                region_path = os.path.join(land_use_path, region_folder)
                if not os.path.isdir(region_path):
                    continue
                for bmp_folder in os.listdir(region_path):
                    bmp_path = os.path.join(region_path, bmp_folder)
                    if os.path.isdir(bmp_path):
                    # Only process a-soy-gb and vr-ha-nt folders
                        if "a-soy-gb-" in bmp_folder or "vr-ha-nt-" in bmp_folder:
                            bmp_folders.append((land_use, region_folder, bmp_folder, bmp_path))
        
        # Cache loaded CSVs by region
        csv_cache = {}
        
        for land_use, region_folder, bmp_folder, bmp_path in tqdm(bmp_folders, desc=f"  {scenario_name}", leave=False):
            
            region_abbrev = get_region_abbrev(region_folder)
            if region_abbrev is None:
                continue
            
            # Load future CSV (with caching)
            if region_abbrev not in csv_cache:
                try:
                    csv_cache[region_abbrev] = load_combined_csv(combined_csv_folder, region_abbrev, scenario)
                except FileNotFoundError:
                    tqdm.write(f"  CSV not found for {region_abbrev}/{scenario}")
                    continue
            
            future_df = csv_cache[region_abbrev]
            
            # Find climate files
            hly_file, dly_file, wp1_file = find_climate_files(bmp_path, region_abbrev)
            
            if hly_file and dly_file and wp1_file:
                try:
                    if variant == "Tonly":
                        # T-only: update temp only, leave precip as baseline
                        update_dly_file(dly_file, future_df)
                        update_wp1_file_tonly(wp1_file, future_df)
                        # .hly stays unchanged (baseline precip)
                    else:  # Ponly
                        # P-only: update precip only, leave temp as baseline
                        update_hly_file(hly_file, future_df)
                        update_dly_file_precip_only(dly_file, future_df)
                        update_wp1_file_ponly(wp1_file, future_df)
                except Exception as e:
                    tqdm.write(f"  ERROR in {bmp_folder}: {e}")
            else:
                missing = []
                if not hly_file: missing.append('.hly')
                if not dly_file: missing.append('.dly')
                if not wp1_file: missing.append('.wp1')
                tqdm.write(f"  Missing files in {bmp_folder}: {missing}")
